# Factory I/O Modbus TCP 제어 노트북 (v5 - Lids/Bases start 동시 실행)

- 대상: Factory I/O (Lids & Bases 조립 시뮬레이션)
- 연결: 210.119.14.58:502, device_id = 1

## v5에서 바뀐 점
- **`pulse_coils()` (복수형) 함수 추가**: 여러 코일을 동시에 ON했다가 동시에 OFF.
  v4까지는 `pulse_coil(LIDS_START)` -> `pulse_coil(BASES_START)`가 **순차 실행**이라
  Lids와 Bases의 start 타이밍이 어긋났음. 이게 "Lids start가 감지되지 않는" 원인으로 추정됨.
- Reset, Start, 최초 Emitter 펄스를 모두 Lids/Bases **동시에** 실행하도록 통일.

## 동작 조건
1. 모든 conveyor 코일 + Remover(Coil18) + 생산모드 플래그(Coil1,8)는 시작 시 ON으로 켜서 유지
2. 에러 상태인 스테이션이 있으면 reset을 동시에 펄스
3. `Lids center (start)`, `Bases center (start)`를 **동시에** 펄스하여 두 스테이션을 같은 시점에 기동
4. 두 emitter를 **동시에** 펄스하여 1개씩 동시 생산
5. exit 센서 하강 엣지마다 카운트, Holding Register 기록, 목표 도달 시 해당 라인 정지


In [7]:
# ===== 1. 라이브러리 및 연결 설정 =====
from pymodbus.client import ModbusTcpClient
import time
import pymodbus

print("설치된 pymodbus 버전:", pymodbus.__version__)

PLC_IP = "210.119.14.58"
PLC_PORT = 502
DEVICE_ID = 1

client = ModbusTcpClient(PLC_IP, port=PLC_PORT)

connected = client.connect()
if not connected:
    raise ConnectionError(
        f"{PLC_IP}:{PLC_PORT} 연결 실패. Factory I/O가 실행 중인지, "
        f"Driver가 Modbus TCP/IP Server로 설정되어 있는지, IP/포트가 맞는지 확인하세요."
    )
print("Factory I/O Modbus TCP 연결 성공")


설치된 pymodbus 버전: 3.13.1
Factory I/O Modbus TCP 연결 성공


In [8]:
# ===== 2. 주소 맵 정의 (0-based, 프로토콜 주소) =====

DISCRETE_INPUTS = {
    0: "Lids at entry", 1: "Lids center (busy)", 2: "Lids center (has error)",
    3: "Lids at exit", 4: "Bases at entry", 5: "Bases center (busy)",
    6: "Bases center (has error)", 7: "Bases at exit", 8: "FACTORY I/O (Running)",
    9: "(라벨 없음 - 미사용/예약)",
}

COILS = {
    0: "Lids raw conveyor", 1: "Lids center (produce lids)", 2: "Lids center (start)",
    3: "Lids center (reset)", 4: "Lids center (stop)", 5: "Lids exit conveyor 1",
    6: "Lids exit conveyor 2", 7: "Bases raw conveyor", 8: "Bases center (produce lids)",
    9: "Bases center (start)", 10: "Bases center (reset)", 11: "Bases center (stop)",
    12: "Bases exit conveyor 1", 13: "Bases exit conveyor 2", 14: "Exit conveyor",
    15: "(라벨 없음 - 미사용/예약)", 16: "Bases emitter", 17: "Lids emitter",
    18: "Remover", 19: "(라벨 없음 - 미사용/예약)",
}

HOLDING_REGS = {0: "Lids counter", 1: "Bases counter"}

LIDS_RAW_CONV      = 0
LIDS_PRODUCE_FLAG  = 1
LIDS_START         = 2
LIDS_RESET         = 3
LIDS_STOP          = 4
LIDS_EXIT_CONV1    = 5
LIDS_EXIT_CONV2    = 6
BASES_RAW_CONV     = 7
BASES_PRODUCE_FLAG = 8
BASES_START        = 9
BASES_RESET        = 10
BASES_STOP         = 11
BASES_EXIT_CONV1   = 12
BASES_EXIT_CONV2   = 13
EXIT_CONV          = 14
BASES_EMITTER      = 16
LIDS_EMITTER       = 17
REMOVER            = 18

LIDS_AT_EXIT_DI     = 3
BASES_AT_EXIT_DI    = 7
LIDS_BUSY_DI        = 1
LIDS_HAS_ERROR_DI   = 2
BASES_BUSY_DI       = 5
BASES_HAS_ERROR_DI  = 6

LIDS_COUNTER_REG   = 0
BASES_COUNTER_REG  = 1

CONVEYOR_COILS = [
    LIDS_RAW_CONV, LIDS_EXIT_CONV1, LIDS_EXIT_CONV2,
    BASES_RAW_CONV, BASES_EXIT_CONV1, BASES_EXIT_CONV2,
    EXIT_CONV,
]
REMOVER_COILS = [REMOVER]
PRODUCE_FLAG_COILS = [LIDS_PRODUCE_FLAG] #, BASES_PRODUCE_FLAG]

# ⚠️ 실행 전에 원하는 목표 수량으로 수정하세요.
TARGET_LIDS_COUNT  = 10
TARGET_BASES_COUNT = 10


In [9]:
# ===== 3. 헬퍼 함수 =====
PULSE_TIME = 0.3

def set_coil(addr, value):
    result = client.write_coil(addr, value, device_id=DEVICE_ID)
    if result.isError():
        print(f"[에러] Coil {addr} 쓰기 실패: {result}")
    return result

def pulse_coil(addr, on_time=PULSE_TIME):
    """코일 1개를 ON했다가 on_time초 후 OFF."""
    name = COILS.get(addr, f"Coil {addr}")
    set_coil(addr, True)
    print(f"[PULSE] {name} (addr {addr}) ON")
    time.sleep(on_time)
    set_coil(addr, False)
    print(f"[PULSE] {name} (addr {addr}) OFF")

def pulse_coils(addrs, on_time=PULSE_TIME):
    """여러 코일을 동시에 ON했다가 동시에 OFF (Lids/Bases 동시 기동용)."""
    for addr in addrs:
        set_coil(addr, True)
        print(f"[PULSE-동시] {COILS.get(addr, addr)} (addr {addr}) ON")
    time.sleep(on_time)
    for addr in addrs:
        set_coil(addr, False)
        print(f"[PULSE-동시] {COILS.get(addr, addr)} (addr {addr}) OFF")

def read_discrete_input(addr):
    result = client.read_discrete_inputs(addr, count=1, device_id=DEVICE_ID)
    if result.isError():
        print(f"[에러] Discrete Input {addr} 읽기 실패: {result}")
        return None
    return result.bits[0]

def read_holding_register(addr):
    result = client.read_holding_registers(addr, count=1, device_id=DEVICE_ID)
    if result.isError():
        print(f"[에러] Holding Register {addr} 읽기 실패: {result}")
        return None
    return result.registers[0]

def write_holding_register(addr, value):
    result = client.write_register(addr, value, device_id=DEVICE_ID)
    if result.isError():
        print(f"[에러] Holding Register {addr} 쓰기 실패: {result}")
    return result


In [10]:
# ===== 4. 진단(디버깅) 셀 - 언제든 안전하게 실행 가능 (읽기 전용) =====
def print_status(label=""):
    print(f"--- 상태 확인 {label} ---")
    print(f"FACTORY I/O Running           : {read_discrete_input(8)}")
    print(f"Lids at entry                  : {read_discrete_input(0)}")
    print(f"Lids center busy                : {read_discrete_input(LIDS_BUSY_DI)}")
    print(f"Lids center has error           : {read_discrete_input(LIDS_HAS_ERROR_DI)}")
    print(f"Lids at exit                    : {read_discrete_input(LIDS_AT_EXIT_DI)}")
    print(f"Bases at entry                  : {read_discrete_input(4)}")
    print(f"Bases center busy               : {read_discrete_input(BASES_BUSY_DI)}")
    print(f"Bases center has error          : {read_discrete_input(BASES_HAS_ERROR_DI)}")
    print(f"Bases at exit                   : {read_discrete_input(BASES_AT_EXIT_DI)}")
    print(f"Lids counter (Holding Reg 0)     : {read_holding_register(LIDS_COUNTER_REG)}")
    print(f"Bases counter (Holding Reg 1)    : {read_holding_register(BASES_COUNTER_REG)}")

print_status("(init_system 실행 전)")


--- 상태 확인 (init_system 실행 전) ---
FACTORY I/O Running           : True
Lids at entry                  : True
Lids center busy                : False
Lids center has error           : False
Lids at exit                    : True
Bases at entry                  : True
Bases center busy               : False
Bases center has error          : False
Bases at exit                   : True
Lids counter (Holding Reg 0)     : 0
Bases counter (Holding Reg 1)    : 0


In [11]:
# ===== 5. 초기화 시퀀스 (조건 1~4) =====
def init_system():
    print("=== 시스템 초기화 시작 ===")

    for addr in CONVEYOR_COILS:
        set_coil(addr, True)
        print(f"[ON] {COILS[addr]} (addr {addr})")

    for addr in REMOVER_COILS:
        set_coil(addr, True)
        print(f"[ON] {COILS[addr]} (addr {addr}) - remover")

    for addr in PRODUCE_FLAG_COILS:
        set_coil(addr, True)
        print(f"[ON] {COILS[addr]} (addr {addr}) - produce flag")

    # v5 핵심 수정: Lids/Bases start를 동시에 ON->동시에 OFF
    set_coil(LIDS_START,1)
    set_coil(BASES_START,1)

    # 에러 상태인 스테이션만 골라서 '동시에' reset 펄스
    reset_targets = []
    if read_discrete_input(LIDS_HAS_ERROR_DI):
        print("[감지] Lids center has error = True -> reset 대상에 포함")
        reset_targets.append(LIDS_RESET)
    if read_discrete_input(BASES_HAS_ERROR_DI):
        print("[감지] Bases center has error = True -> reset 대상에 포함")
        reset_targets.append(BASES_RESET)
    if reset_targets:
        pulse_coils(reset_targets, on_time=0.5)

    write_holding_register(LIDS_COUNTER_REG, 0)
    write_holding_register(BASES_COUNTER_REG, 0)
    print("[INIT] 카운터 레지스터 초기화 완료 (Lids=0, Bases=0)")

    # 최초 생산도 동시에
    pulse_coils([LIDS_EMITTER, BASES_EMITTER])

    print("=== 시스템 초기화 완료 ===")

init_system()
print_status("(init_system 실행 후)")


=== 시스템 초기화 시작 ===
[ON] Lids raw conveyor (addr 0)
[ON] Lids exit conveyor 1 (addr 5)
[ON] Lids exit conveyor 2 (addr 6)
[ON] Bases raw conveyor (addr 7)
[ON] Bases exit conveyor 1 (addr 12)
[ON] Bases exit conveyor 2 (addr 13)
[ON] Exit conveyor (addr 14)
[ON] Remover (addr 18) - remover
[ON] Lids center (produce lids) (addr 1) - produce flag
[INIT] 카운터 레지스터 초기화 완료 (Lids=0, Bases=0)
[PULSE-동시] Lids emitter (addr 17) ON
[PULSE-동시] Bases emitter (addr 16) ON
[PULSE-동시] Lids emitter (addr 17) OFF
[PULSE-동시] Bases emitter (addr 16) OFF
=== 시스템 초기화 완료 ===
--- 상태 확인 (init_system 실행 후) ---
FACTORY I/O Running           : True
Lids at entry                  : True
Lids center busy                : False
Lids center has error           : False
Lids at exit                    : True
Bases at entry                  : True
Bases center busy               : False
Bases center has error          : False
Bases at exit                   : True
Lids counter (Holding Reg 0)     : 0
Bases counter (Holdi

In [12]:
# ===== 6. 메인 감시 루프 (조건 5) =====
def run_loop(poll_interval=0.1):
    prev_lids_exit = read_discrete_input(LIDS_AT_EXIT_DI)
    prev_bases_exit = read_discrete_input(BASES_AT_EXIT_DI)

    lids_count = 0
    bases_count = 0
    lids_done = False
    bases_done = False

    print(f"메인 루프 시작 (목표: Lids={TARGET_LIDS_COUNT}, Bases={TARGET_BASES_COUNT})")
    print("중지하려면 Jupyter 상단의 Interrupt/Stop 버튼 사용")

    try:
        while True:
            cur_lids_exit = read_discrete_input(LIDS_AT_EXIT_DI)
            cur_bases_exit = read_discrete_input(BASES_AT_EXIT_DI)

            

            if prev_lids_exit is True and cur_lids_exit is False:
                lids_count += 1
                write_holding_register(LIDS_COUNTER_REG, lids_count)
                print(f">> Lids exit 통과 감지 (누적 {lids_count}/{TARGET_LIDS_COUNT})")
                if lids_count >= TARGET_LIDS_COUNT and not lids_done:
                    lids_done = True
                    set_coil(LIDS_RAW_CONV, False)
                    print(f"[목표 도달] Lids {TARGET_LIDS_COUNT}개 완료 -> Lids raw conveyor 정지")
                elif not lids_done:
                    pulse_coil(LIDS_EMITTER)

            if prev_bases_exit is True and cur_bases_exit is False:
                bases_count += 1
                write_holding_register(BASES_COUNTER_REG, bases_count)
                print(f">> Bases exit 통과 감지 (누적 {bases_count}/{TARGET_BASES_COUNT})")
                if bases_count >= TARGET_BASES_COUNT and not bases_done:
                    bases_done = True
                    set_coil(BASES_RAW_CONV, False)
                    print(f"[목표 도달] Bases {TARGET_BASES_COUNT}개 완료 -> Bases raw conveyor 정지")
                elif not bases_done:
                    pulse_coil(BASES_EMITTER)

            prev_lids_exit = cur_lids_exit
            prev_bases_exit = cur_bases_exit

            if lids_done and bases_done:
                print("=== 양쪽 목표 수량 모두 도달. 메인 루프를 종료합니다. ===")
                break

            time.sleep(poll_interval)

    except KeyboardInterrupt:
        print("사용자가 루프를 중지했습니다.")

    return lids_count, bases_count

final_lids, final_bases = run_loop()
print(f"최종 생산량 - Lids: {final_lids}, Bases: {final_bases}")


메인 루프 시작 (목표: Lids=10, Bases=10)
중지하려면 Jupyter 상단의 Interrupt/Stop 버튼 사용
>> Bases exit 통과 감지 (누적 1/10)
[PULSE] Bases emitter (addr 16) ON
[PULSE] Bases emitter (addr 16) OFF
>> Lids exit 통과 감지 (누적 1/10)
[PULSE] Lids emitter (addr 17) ON
[PULSE] Lids emitter (addr 17) OFF
>> Bases exit 통과 감지 (누적 2/10)
[PULSE] Bases emitter (addr 16) ON
[PULSE] Bases emitter (addr 16) OFF
>> Lids exit 통과 감지 (누적 2/10)
[PULSE] Lids emitter (addr 17) ON
[PULSE] Lids emitter (addr 17) OFF
>> Bases exit 통과 감지 (누적 3/10)
[PULSE] Bases emitter (addr 16) ON
[PULSE] Bases emitter (addr 16) OFF
>> Lids exit 통과 감지 (누적 3/10)
[PULSE] Lids emitter (addr 17) ON
[PULSE] Lids emitter (addr 17) OFF
>> Bases exit 통과 감지 (누적 4/10)
[PULSE] Bases emitter (addr 16) ON
[PULSE] Bases emitter (addr 16) OFF
>> Lids exit 통과 감지 (누적 4/10)
[PULSE] Lids emitter (addr 17) ON
[PULSE] Lids emitter (addr 17) OFF
>> Bases exit 통과 감지 (누적 5/10)
[PULSE] Bases emitter (addr 16) ON
[PULSE] Bases emitter (addr 16) OFF
>> Lids exit 통과 감지 (누적 5/10)
[P

In [65]:
# ===== 7. 안전 정지 (필요할 때 이 셀만 따로 실행) =====
def stop_all():
    for addr in COILS:
        set_coil(addr, False)
    print("모든 코일 OFF 완료")

stop_all()
client.close()
print("Modbus 연결 종료")


모든 코일 OFF 완료
Modbus 연결 종료
